In [1]:
import pandas as pd
import numpy as np

In [2]:
# Total number of insurance policies to simulate
n = 1000000


In [3]:
#  Generate policy tenure based on required distribution
# 20% → 1 year, 30% → 2 years, 40% → 3 years, 10% → 4 years
tenure = np.random.choice(
    [1,2,3,4],
    size=n,
    p=[0.2,0.3,0.4,0.1]
)

In [4]:
dates = pd.date_range("2024-01-01","2024-12-31")

purchase_dates = np.random.choice(dates, n)

In [5]:
policy_df = pd.DataFrame({
    "Customer_ID": range(1, n+1),
    "Vehicle_ID": range(1000001, 1000001+n),
    "Vehicle_Value": 100000,
    "Policy_Tenure": tenure,
    "Policy_Purchase_Date": purchase_dates
})

In [6]:
policy_df["Policy_Start_Date"] = policy_df["Policy_Purchase_Date"] + pd.Timedelta(days=365)

In [7]:
policy_df["Policy_End_Date"] = policy_df["Policy_Start_Date"] + pd.to_timedelta(policy_df["Policy_Tenure"]*365, unit="d")

In [8]:
# Premium = ₹100 per year of policy tenure
policy_df["Premium"] = policy_df["Policy_Tenure"] * 100

In [9]:
policy_df.head()

,Customer_ID,Vehicle_ID,Vehicle_Value,Policy_Tenure,Policy_Purchase_Date,Policy_Start_Date,Policy_End_Date,Premium
0,1,1000001,100000,3,2024-10-24,2025-10-24,2028-10-23,300
1,2,1000002,100000,1,2024-05-19,2025-05-19,2026-05-19,100
2,3,1000003,100000,3,2024-02-18,2025-02-17,2028-02-17,300
3,4,1000004,100000,1,2024-08-22,2025-08-22,2026-08-22,100
4,5,1000005,100000,3,2024-10-11,2025-10-11,2028-10-10,300


In [10]:
policy_df.to_csv("policy_sales_data.csv", index=False)

In [11]:
policy_df.shape

(1000000, 8)

In [12]:
policy_df["Purchase_Day"] = policy_df["Policy_Purchase_Date"].dt.day

In [13]:
eligible_2025 = policy_df[
    policy_df["Purchase_Day"].isin([7,14,21,28])
]

In [14]:
claims_2025 = eligible_2025.sample(frac=0.3, random_state=42)

In [15]:
claims_2025_df = pd.DataFrame({
    "Customer_ID": claims_2025["Customer_ID"],
    "Vehicle_ID": claims_2025["Vehicle_ID"],
    "Claim_Date": claims_2025["Policy_Start_Date"],
    "Claim_Amount": 10000,
    "Claim_Type": 1
})

In [16]:
claims_2025_df["Claim_ID"] = range(1, len(claims_2025_df)+1)

In [17]:
claims_2025_df = claims_2025_df[
    ["Claim_ID","Customer_ID","Vehicle_ID","Claim_Amount","Claim_Date","Claim_Type"]
]

In [18]:
four_year_policies = policy_df[
    policy_df["Policy_Tenure"] == 4
]

In [19]:
claims_2026 = four_year_policies.sample(frac=0.1, random_state=42)

In [20]:
claim_dates_2026 = pd.date_range("2026-01-01","2026-02-28")

In [21]:
claims_2026["Claim_Date"] = np.random.choice(claim_dates_2026, len(claims_2026))

In [22]:
claims_2026_df = pd.DataFrame({
    "Customer_ID": claims_2026["Customer_ID"],
    "Vehicle_ID": claims_2026["Vehicle_ID"],
    "Claim_Date": claims_2026["Claim_Date"],
    "Claim_Amount": 10000,
    "Claim_Type": 2
})

In [23]:
claims_2026_df["Claim_ID"] = range(
    len(claims_2025_df)+1,
    len(claims_2025_df)+len(claims_2026_df)+1
)

In [24]:
claims_df = pd.concat([claims_2025_df, claims_2026_df])

In [25]:
claims_df.head()

,Claim_ID,Customer_ID,Vehicle_ID,Claim_Amount,Claim_Date,Claim_Type
495792,1,495793,1495793,10000,2025-12-28,1
642000,2,642001,1642001,10000,2025-11-14,1
348415,3,348416,1348416,10000,2025-03-14,1
725732,4,725733,1725733,10000,2025-07-14,1
834805,5,834806,1834806,10000,2025-10-07,1


In [26]:
claims_df.to_csv("claims_data.csv", index=False)

In [27]:
claims_df.shape

(49173, 6)

In [28]:
# total premium is simply the sum 
total_premium = policy_df["Premium"].sum()

print("Total Premium Collected in 2024:", total_premium)

Total Premium Collected in 2024: 239916200


In [29]:
claims_df["Year"] = pd.to_datetime(claims_df["Claim_Date"]).dt.year
claims_df["Month"] = pd.to_datetime(claims_df["Claim_Date"]).dt.month

In [30]:
monthly_claims = claims_df.groupby(["Year","Month"])["Claim_Amount"].sum().reset_index()

monthly_claims

,Year,Month,Claim_Amount
0,2025,1,33110000
1,2025,2,31690000
2,2025,3,33080000
3,2025,4,32840000
4,2025,5,32690000
5,2025,6,33120000
6,2025,7,31650000
7,2025,8,33160000
8,2025,9,32660000
9,2025,10,32100000


In [31]:
merged_df = pd.merge(
    claims_df,
    policy_df[["Customer_ID","Policy_Tenure","Premium"]],
    on="Customer_ID"
)

In [32]:
ratio_tenure = merged_df.groupby("Policy_Tenure").agg({
"Claim_Amount":"sum",
"Premium":"sum"
})

ratio_tenure["Loss_Ratio"] = ratio_tenure["Claim_Amount"] / ratio_tenure["Premium"]

ratio_tenure

,Claim_Amount,Premium,Loss_Ratio
Policy_Tenure,,,
1,77000000,770000,100.000000
2,118410000,2368200,50.000000
3,157500000,4725000,33.333333
4,138820000,5552800,25.000000


In [33]:
policy_df["Sale_Month"] = pd.to_datetime(policy_df["Policy_Purchase_Date"]).dt.month

In [34]:
merged_df2 = pd.merge(
claims_df,
policy_df[["Customer_ID","Sale_Month","Premium"]],
on="Customer_ID"
)

In [35]:
ratio_month = merged_df2.groupby("Sale_Month").agg({
"Claim_Amount":"sum",
"Premium":"sum"
})

ratio_month["Loss_Ratio"] = ratio_month["Claim_Amount"]/ratio_month["Premium"]

ratio_month

,Claim_Amount,Premium,Loss_Ratio
Sale_Month,,,
1,41390000,1125800,36.764967
2,39750000,1090400,36.454512
3,41870000,1150400,36.396036
4,40850000,1111700,36.745525
5,41410000,1139800,36.330935
6,41310000,1131300,36.515513
7,40100000,1099700,36.464490
8,41660000,1131000,36.834660
9,40620000,1104300,36.783483


In [36]:
# Vehicle without claims will eventually claim once

In [37]:
vehicles_with_claims = claims_df["Vehicle_ID"].nunique()

total_vehicles = policy_df["Vehicle_ID"].nunique()

remaining_vehicles = total_vehicles - vehicles_with_claims

In [38]:
potential_liability = remaining_vehicles * 10000

print("Estimated Future Claim Liability:", potential_liability)

Estimated Future Claim Liability: 9511880000


In [39]:
# earn Premium Until Feb 28  2026
# Daily Premium = Premium / Tenure Days

In [40]:
policy_df["Tenure_Days"] = policy_df["Policy_Tenure"] * 365

In [41]:
policy_df["Daily_Premium"] = policy_df["Premium"] / policy_df["Tenure_Days"]

In [42]:
cutoff_date = pd.to_datetime("2026-02-28")

policy_df["Earned_Days"] = (cutoff_date - policy_df["Policy_Start_Date"]).dt.days
policy_df["Earned_Days"] = policy_df["Earned_Days"].clip(lower=0)

policy_df["Earned_Premium"] = policy_df["Earned_Days"] * policy_df["Daily_Premium"]

In [43]:
# Total Earned premium 

In [44]:
earned_premium = policy_df["Earned_Premium"].sum()

print("Earned Premium until Feb 28 2026:", earned_premium)

Earned Premium until Feb 28 2026: 66158801.36986301


In [45]:
policy_df.to_csv("policy_sales_data.csv", index=False)
claims_df.to_csv("claims_data.csv", index=False)

print("Files saved successfully")

Files saved successfully


In [47]:
pd.read_csv("policy_sales_data.csv").head()

,Customer_ID,Vehicle_ID,Vehicle_Value,Policy_Tenure,Policy_Purchase_Date,Policy_Start_Date,Policy_End_Date,Premium,Purchase_Day,Sale_Month,Tenure_Days,Daily_Premium,Earned_Days,Earned_Premium
0,1,1000001,100000,3,2024-10-24,2025-10-24,2028-10-23,300,24,10,1095,0.273973,127,34.794521
1,2,1000002,100000,1,2024-05-19,2025-05-19,2026-05-19,100,19,5,365,0.273973,285,78.082192
2,3,1000003,100000,3,2024-02-18,2025-02-17,2028-02-17,300,18,2,1095,0.273973,376,103.013699
3,4,1000004,100000,1,2024-08-22,2025-08-22,2026-08-22,100,22,8,365,0.273973,190,52.054795
4,5,1000005,100000,3,2024-10-11,2025-10-11,2028-10-10,300,11,10,1095,0.273973,140,38.356164
